# FallGuardNet v2 — Knowledge Distillation from ResNet18

**Strategy:** Train ResNet18 (teacher, 100% specificity) → Distill into a redesigned lightweight FallGuardNet v2 (student, ~32K params)

**Goal:** Beat MobileNetV3's F1=0.75 and Specificity=0.9167 while staying under 40K params

In [ ]:
!pip install -q einops onnx
!git clone https://github.com/AutoSmartGroup/ESP-Fi-HAR.git

import os, glob, time, json, random, warnings
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 2: Dataset with Heavy Augmentation for Falls
# ══════════════════════════════════════════════════════

DATA_ROOT = '/kaggle/working/ESP-Fi-HAR/Model Code/Data'
ACTIVITIES = ['arm_wave', 'fall', 'jump', 'run', 'squat', 'turn', 'walk']

def load_mat_files(split):
    """Load all .mat files, return (samples, binary_labels, activity_names)"""
    data, labels, activities = [], [], []
    root = os.path.join(DATA_ROOT, split)
    for act in ACTIVITIES:
        files = sorted(glob.glob(os.path.join(root, act, '*.mat')))
        for f in files:
            try:
                x = sio.loadmat(f)['CSIamp']
                if x.shape == (52, 950): x = x.T
                if x.shape != (950, 52): continue
                x = x.astype(np.float32)
                # Z-score normalize each sample independently
                x = (x - np.mean(x)) / (np.std(x) + 1e-8)
                data.append(x)
                labels.append(1 if act == 'fall' else 0)
                activities.append(act)
            except Exception as e:
                print(f'Error loading {f}: {e}')
    return np.array(data), np.array(labels, dtype=np.int64), activities


class FallDataset(Dataset):
    def __init__(self, split='train_amp', augment=False):
        self.augment = augment
        raw_data, raw_labels, self.activities = load_mat_files(split)
        
        if augment:
            # Augment fall samples: time-reverse + time-warp copies
            aug_data, aug_labels, aug_acts = [], [], []
            for i in range(len(raw_data)):
                aug_data.append(raw_data[i])
                aug_labels.append(raw_labels[i])
                aug_acts.append(self.activities[i])
                
                if raw_labels[i] == 1:  # Fall sample
                    # Time-reversed copy
                    aug_data.append(raw_data[i][::-1].copy())
                    aug_labels.append(1)
                    aug_acts.append('fall')
                    # Slightly time-warped copy
                    shifted = np.roll(raw_data[i], np.random.randint(-80, 80), axis=0)
                    aug_data.append(shifted)
                    aug_labels.append(1)
                    aug_acts.append('fall')
            
            self.data = np.array(aug_data)
            self.labels = np.array(aug_labels, dtype=np.int64)
            self.activities = aug_acts
        else:
            self.data = raw_data
            self.labels = raw_labels
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx].copy()
        
        if self.augment:
            # Random time shift
            shift = np.random.randint(-100, 100)
            if shift: x = np.roll(x, shift, axis=0)
            # Amplitude scaling
            x *= np.random.uniform(0.85, 1.15)
            # Gaussian noise
            x += np.random.normal(0, 0.02, x.shape).astype(np.float32)
        
        return torch.tensor(x, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


# Build datasets
train_ds = FallDataset('train_amp', augment=True)
test_ds  = FallDataset('test_amp',  augment=False)

# Weighted sampler for class imbalance
counts = Counter(train_ds.labels.tolist())
weights = {c: len(train_ds) / (2.0 * n) for c, n in counts.items()}
sample_weights = [weights[int(l)] for l in train_ds.labels]
sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples | Distribution: {dict(counts)}')
print(f'Test:  {len(test_ds)} samples')
print(f'Class weights: {weights}')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 3: Teacher Model — ResNet18 (11M params, 100% specificity)
# ══════════════════════════════════════════════════════

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_ch, out_ch, stride=(1,1), downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class TeacherResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.in_ch = 64
        self.preprocess = nn.Sequential(
            nn.Conv2d(1, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.layer1 = self._make_layer(64,  2, stride=(1,1))
        self.layer2 = self._make_layer(128, 2, stride=(2,1))
        self.layer3 = self._make_layer(256, 2, stride=(2,1))
        self.layer4 = self._make_layer(512, 2, stride=(2,1))
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_ch, blocks, stride):
        downsample = None
        if stride != (1,1) or self.in_ch != out_ch:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        layers = [BasicBlock(self.in_ch, out_ch, stride, downsample)]
        self.in_ch = out_ch
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, 950, 52) -> reshape to (B, 1, 950, 52)
        if x.dim() == 3: x = x.unsqueeze(1)
        x = self.preprocess(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1))

teacher = TeacherResNet18().to(device)
print(f'Teacher (ResNet18) params: {sum(p.numel() for p in teacher.parameters()):,}')

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 4: Train the Teacher to 100% Specificity
# ══════════════════════════════════════════════════════

print('='*60)
print('PHASE 1: Training Teacher (ResNet18)')
print('='*60)

class_weights = torch.tensor([weights[0], weights[1]], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(teacher.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=60)

best_teacher_f1 = 0
teacher_history = {'train_loss': [], 'test_acc': [], 'test_f1': []}

for epoch in range(60):
    teacher.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(teacher(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    
    # Evaluate
    teacher.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds = teacher(x.to(device)).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(y.numpy())
    
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    f1 = f1_score(all_labels, all_preds, pos_label=1)
    teacher_history['train_loss'].append(total_loss / len(train_loader))
    teacher_history['test_acc'].append(acc)
    teacher_history['test_f1'].append(f1)
    
    if f1 > best_teacher_f1:
        best_teacher_f1 = f1
        torch.save(teacher.state_dict(), 'teacher_resnet18_best.pt')
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:02d}/60 | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}')

# Load best teacher
teacher.load_state_dict(torch.load('teacher_resnet18_best.pt'))
teacher.eval()

# Final teacher eval
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds = teacher(x.to(device)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
print(f'\n✅ Teacher Final: Acc={np.mean(np.array(all_preds)==np.array(all_labels)):.4f} | '
      f'F1={f1_score(all_labels, all_preds, pos_label=1):.4f} | '
      f'Specificity={tn/(tn+fp):.4f} | Recall={tp/(tp+fn):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 5: FallGuardNet v2 — Redesigned Student Architecture
# ══════════════════════════════════════════════════════════════════════
#
# KEY DESIGN CHANGES vs v1:
#   1. Periodicity Detector: Explicitly models whether motion is periodic
#      (walking/running) vs aperiodic (falls). This is the #1 reason v1
#      confused walks with falls.
#   2. Multi-Resolution Temporal Conv: Three parallel branches at different
#      temporal scales (fine=3, medium=7, coarse=15) to capture both the
#      sharp impact spike of a fall AND the smooth gait cycle of walking.
#   3. Channel SE Attention: Learns which of the 52 subcarriers are most
#      discriminative for fall vs non-fall.
#   4. Stillness Detector: Falls end with stillness. Walking doesn't.
#      A dedicated branch measures tail-end motion energy.

class DepthwiseSepConv1d(nn.Module):
    """Efficient depthwise separable convolution."""
    def __init__(self, c_in, c_out, k, dilation=1):
        super().__init__()
        pad = (k - 1) * dilation // 2
        self.dw = nn.Conv1d(c_in, c_in, k, padding=pad, dilation=dilation, groups=c_in, bias=False)
        self.pw = nn.Conv1d(c_in, c_out, 1)
        self.bn = nn.BatchNorm1d(c_out)
    def forward(self, x):
        return F.gelu(self.bn(self.pw(self.dw(x))))

class ChannelSE(nn.Module):
    """Squeeze-and-Excitation on channels."""
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(c, c // r), nn.ReLU(),
            nn.Linear(c // r, c), nn.Sigmoid()
        )
    def forward(self, x):
        # x: (B, C, T)
        s = x.mean(-1)  # (B, C)
        return x * self.fc(s).unsqueeze(-1)


class FallGuardNetV2(nn.Module):
    """
    FallGuardNet v2: Multi-Resolution Temporal Network with
    Periodicity Awareness and Stillness Detection.
    
    Architecture:
      Input (B, 950, 52)
        │
        ├─→ Fine Branch   (k=3, dilations 1,2,4)     → captures sharp spikes
        ├─→ Medium Branch (k=7, dilations 1,2,4)     → captures motion patterns  
        └─→ Coarse Branch (k=15, dilations 1,2)      → captures slow trends
        │
        └─→ Concatenate + Channel SE Attention
        │
        ├─→ Temporal Attention Pool → Main classifier head
        ├─→ Periodicity Score (autocorrelation feature)
        └─→ Stillness Score (tail energy feature)
    """
    
    def __init__(self, c=32, num_classes=2):
        super().__init__()
        sub = 52  # ESP32 subcarriers
        
        # === Multi-Resolution Branches ===
        # Fine: sharp temporal features (impact detection)
        self.fine = nn.Sequential(
            DepthwiseSepConv1d(sub, c, k=3, dilation=1),
            DepthwiseSepConv1d(c, c, k=3, dilation=2),
            DepthwiseSepConv1d(c, c, k=3, dilation=4),
        )
        # Medium: motion pattern features (gait cycle)
        self.medium = nn.Sequential(
            DepthwiseSepConv1d(sub, c, k=7, dilation=1),
            DepthwiseSepConv1d(c, c, k=7, dilation=2),
            DepthwiseSepConv1d(c, c, k=7, dilation=4),
        )
        # Coarse: slow trend features (posture change)
        self.coarse = nn.Sequential(
            DepthwiseSepConv1d(sub, c, k=15, dilation=1),
            DepthwiseSepConv1d(c, c, k=15, dilation=2),
        )
        
        # === Fusion ===
        self.fuse = nn.Sequential(
            nn.Conv1d(c * 3, c, 1, bias=False),
            nn.BatchNorm1d(c),
            nn.GELU()
        )
        self.se = ChannelSE(c, r=4)
        
        # === Temporal Attention Pooling ===
        self.attn_score = nn.Conv1d(c, 1, 1)
        
        # === Classification Head ===
        # Features: attn_pool(c) + max_pool(c) + periodicity(1) + stillness(1)
        feat_dim = c * 2 + 2
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, num_classes)
        )
    
    def _periodicity_score(self, x_raw):
        """Compute autocorrelation-based periodicity. Walking/running = high, fall = low."""
        # x_raw: (B, 950, 52)
        energy = x_raw.pow(2).mean(-1)  # (B, 950)
        # Normalized autocorrelation at lag ~100 (roughly 1 step cycle at 100Hz)
        T = energy.shape[1]
        lag = 100
        if T > lag:
            e1 = energy[:, :T-lag]
            e2 = energy[:, lag:]
            # Pearson correlation
            e1_centered = e1 - e1.mean(-1, keepdim=True)
            e2_centered = e2 - e2.mean(-1, keepdim=True)
            num = (e1_centered * e2_centered).sum(-1)
            den = (e1_centered.pow(2).sum(-1).sqrt() * e2_centered.pow(2).sum(-1).sqrt()) + 1e-8
            return (num / den).unsqueeze(-1)  # (B, 1)
        return torch.zeros(x_raw.shape[0], 1, device=x_raw.device)
    
    def _stillness_score(self, x_raw):
        """Measure how still the last 20% of the signal is. Falls end still, walks don't."""
        # x_raw: (B, 950, 52)
        T = x_raw.shape[1]
        tail = x_raw[:, int(T*0.8):, :]  # Last 20%
        body = x_raw[:, :int(T*0.8), :]  # First 80%
        
        tail_energy = tail.diff(dim=1).pow(2).mean(dim=(1,2))  # Motion in tail
        body_energy = body.diff(dim=1).pow(2).mean(dim=(1,2))  # Motion in body
        
        # Ratio: high means tail is much calmer than body (= fall pattern)
        stillness = 1.0 - (tail_energy / (body_energy + 1e-8)).clamp(0, 1)
        return stillness.unsqueeze(-1)  # (B, 1)
    
    def forward(self, x):
        # x: (B, 950, 52)
        x_raw = x
        x_t = x.transpose(1, 2)  # (B, 52, 950)
        
        # Multi-resolution features
        f_fine   = self.fine(x_t)     # (B, c, 950)
        f_medium = self.medium(x_t)   # (B, c, 950)
        f_coarse = self.coarse(x_t)   # (B, c, T') — might differ slightly
        
        # Align temporal dims
        T = f_fine.shape[-1]
        f_medium = F.interpolate(f_medium, size=T, mode='linear', align_corners=False)
        f_coarse = F.interpolate(f_coarse, size=T, mode='linear', align_corners=False)
        
        # Fuse + SE attention
        fused = self.fuse(torch.cat([f_fine, f_medium, f_coarse], dim=1))  # (B, c, T)
        fused = self.se(fused)
        
        # Temporal attention pooling
        attn_w = torch.softmax(self.attn_score(fused), dim=-1)  # (B, 1, T)
        attn_pool = (fused * attn_w).sum(-1)  # (B, c)
        max_pool = fused.amax(-1)              # (B, c)
        
        # Physics-informed features
        periodicity = self._periodicity_score(x_raw)  # (B, 1)
        stillness   = self._stillness_score(x_raw)    # (B, 1)
        
        # Concatenate all features
        feat = torch.cat([attn_pool, max_pool, periodicity, stillness], dim=1)
        
        return self.classifier(feat)


student = FallGuardNetV2(c=32).to(device)
n_params = sum(p.numel() for p in student.parameters())
print(f'FallGuardNet v2 params: {n_params:,}')

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, 950, 52).to(device)
    out = student(dummy)
    print(f'Output shape: {out.shape}')  # Should be (2, 2)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 6: Knowledge Distillation Training
# ══════════════════════════════════════════════════════════════════════
#
# Loss = alpha * KD_loss(student_soft, teacher_soft) + (1-alpha) * CE_loss(student, labels)
# where soft = softmax(logits / temperature)

def distillation_loss(student_logits, teacher_logits, labels, 
                       class_weights, alpha=0.7, temperature=4.0):
    """Combined knowledge distillation + hard label loss."""
    # Soft targets from teacher
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    kd_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (temperature ** 2)
    
    # Hard target CE loss with class weights
    ce_loss = F.cross_entropy(student_logits, labels, weight=class_weights)
    
    return alpha * kd_loss + (1 - alpha) * ce_loss


print('='*60)
print('PHASE 2: Knowledge Distillation (ResNet18 → FallGuardNet v2)')
print('='*60)

teacher.eval()  # Freeze teacher
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=100)

best_student_f1 = 0
student_history = {'train_loss': [], 'test_acc': [], 'test_f1': [], 
                   'test_prec': [], 'test_rec': [], 'test_spec': []}

for epoch in range(100):
    student.train()
    total_loss = 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        # Get teacher predictions (no grad)
        with torch.no_grad():
            teacher_logits = teacher(x)
        
        # Student forward
        student_logits = student(x)
        
        # Distillation loss
        loss = distillation_loss(student_logits, teacher_logits, y,
                                 class_weights, alpha=0.7, temperature=4.0)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    
    # Evaluate
    student.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds = student(x.to(device)).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(y.numpy())
    
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    f1  = f1_score(all_labels, all_preds, pos_label=1)
    pre = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
    rec = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    cm  = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    student_history['train_loss'].append(total_loss / len(train_loader))
    student_history['test_acc'].append(acc)
    student_history['test_f1'].append(f1)
    student_history['test_prec'].append(pre)
    student_history['test_rec'].append(rec)
    student_history['test_spec'].append(spec)
    
    if f1 > best_student_f1:
        best_student_f1 = f1
        torch.save(student.state_dict(), 'fallguardnet_v2_best.pt')
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:03d}/100 | Loss: {total_loss/len(train_loader):.4f} | '
              f'Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {pre:.4f} | Rec: {rec:.4f} | Spec: {spec:.4f}')

# Load best
student.load_state_dict(torch.load('fallguardnet_v2_best.pt'))
print(f'\n✅ Best Student F1: {best_student_f1:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 7: Final Evaluation + Per-Activity False Positive Analysis
# ══════════════════════════════════════════════════════════════════════

student.eval()
all_preds, all_labels, all_acts = [], [], []

with torch.no_grad():
    for i in range(len(test_ds)):
        x, y = test_ds[i]
        pred = student(x.unsqueeze(0).to(device)).argmax(1).item()
        all_preds.append(pred)
        all_labels.append(y.item())
        all_acts.append(test_ds.activities[i])

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Overall metrics
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
acc  = np.mean(all_preds == all_labels)
f1   = f1_score(all_labels, all_preds, pos_label=1)
prec = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
rec  = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
spec = tn / (tn + fp)

print('═'*60)
print('FINAL RESULTS: FallGuardNet v2 (Distilled)')
print('═'*60)
print(f'  Accuracy:    {acc:.4f}')
print(f'  Precision:   {prec:.4f}')
print(f'  Recall:      {rec:.4f}')
print(f'  F1-Score:    {f1:.4f}')
print(f'  Specificity: {spec:.4f}')
print(f'  Parameters:  {n_params:,}')
print(f'\n  Confusion Matrix:')
print(f'    TN={tn}  FP={fp}')
print(f'    FN={fn}  TP={tp}')

# Per-activity false positive rate
print(f'\n  Per-Activity False Positive Rate (non-fall predicted as fall):')
fp_rates = {}
for act in ACTIVITIES:
    if act == 'fall': continue
    mask = [a == act for a in all_acts]
    act_preds = all_preds[mask]
    fpr = act_preds.sum() / len(act_preds) if len(act_preds) > 0 else 0
    fp_rates[act] = fpr
    status = '✅' if fpr == 0 else '⚠️' if fpr < 0.3 else '❌'
    print(f'    {status} {act:12s}: {fpr*100:.1f}% ({int(act_preds.sum())}/{len(act_preds)} misclassified)')

# Comparison table
print(f'\n\n═══ COMPARISON WITH PREVIOUS BENCHMARK ═══')
print(f'| Model            | Params   | Acc    | Prec   | Recall | F1     | Spec   |')
print(f'|------------------|----------|--------|--------|--------|--------|--------|')
print(f'| MobileNetV3      | 36,322   | 0.9143 | 0.6429 | 0.9000 | 0.7500 | 0.9167 |')
print(f'| FallGuardNet v1  | 32,260   | 0.8714 | 0.5294 | 0.9000 | 0.6667 | 0.8667 |')
print(f'| **FallGuardNet v2** | **{n_params:,}** | **{acc:.4f}** | **{prec:.4f}** | **{rec:.4f}** | **{f1:.4f}** | **{spec:.4f}** |')
print(f'| Teacher ResNet18 | 11.1M    | 0.9571 | 1.0000 | 0.7000 | 0.8235 | 1.0000 |')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 8: Visualization
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('FallGuardNet v2 — Knowledge Distillation Results', fontsize=16, fontweight='bold')

# 1. Training Loss
axes[0,0].plot(student_history['train_loss'], color='#e74c3c', linewidth=1.5)
axes[0,0].set_title('Training Loss (Distillation)')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].grid(True, alpha=0.3)

# 2. Test Metrics over epochs
axes[0,1].plot(student_history['test_f1'], label='F1', linewidth=1.5)
axes[0,1].plot(student_history['test_spec'], label='Specificity', linewidth=1.5)
axes[0,1].plot(student_history['test_rec'], label='Recall', linewidth=1.5)
axes[0,1].set_title('Test Metrics Over Training')
axes[0,1].set_xlabel('Epoch'); axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Confusion Matrix
im = axes[0,2].imshow(cm, cmap='Blues')
axes[0,2].set_title('Confusion Matrix')
axes[0,2].set_xticks([0,1]); axes[0,2].set_yticks([0,1])
axes[0,2].set_xticklabels(['Non-Fall', 'Fall'])
axes[0,2].set_yticklabels(['Non-Fall', 'Fall'])
axes[0,2].set_xlabel('Predicted'); axes[0,2].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[0,2].text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=20, fontweight='bold')

# 4. Per-Activity False Positive Rate
act_names = list(fp_rates.keys())
act_vals  = [fp_rates[a]*100 for a in act_names]
colors = ['#27ae60' if v == 0 else '#e67e22' if v < 30 else '#e74c3c' for v in act_vals]
axes[1,0].bar(act_names, act_vals, color=colors)
axes[1,0].set_title('False Positive Rate by Activity (%)')
axes[1,0].set_ylabel('FP Rate (%)')
axes[1,0].axhline(y=0, color='green', linestyle='--', alpha=0.5)
for i, v in enumerate(act_vals):
    axes[1,0].text(i, v+1, f'{v:.0f}%', ha='center', fontweight='bold')

# 5. Benchmark Comparison — F1
models = ['MLP', 'CNN', 'GRU', 'LSTM', 'Transformer', 'MobileNetV3', 'FGN v1', 'FGN v2\n(Ours)']
f1s =    [0.4545, 0.5625, 0.4762, 0.3214, 0.6667, 0.7500, 0.6667, f1]
bar_colors = ['#95a5a6']*6 + ['#e74c3c', '#27ae60']
axes[1,1].bar(models, f1s, color=bar_colors)
axes[1,1].set_title('F1-Score Comparison')
axes[1,1].set_ylabel('F1-Score')
for i, v in enumerate(f1s):
    axes[1,1].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')
axes[1,1].tick_params(axis='x', labelsize=8)

# 6. Benchmark Comparison — Specificity
specs = [0.8833, 0.7833, 0.9000, 0.3833, 0.8667, 0.9167, 0.8667, spec]
axes[1,2].bar(models, specs, color=bar_colors)
axes[1,2].set_title('Specificity Comparison (Higher = Fewer False Alarms)')
axes[1,2].set_ylabel('Specificity')
for i, v in enumerate(specs):
    axes[1,2].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')
axes[1,2].tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig('fallguardnet_v2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📊 Saved: fallguardnet_v2_results.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 9: Export to ONNX + Save Results
# ══════════════════════════════════════════════════════════════════════

student.eval()
student_cpu = FallGuardNetV2(c=32)
student_cpu.load_state_dict(torch.load('fallguardnet_v2_best.pt', map_location='cpu'))
student_cpu.eval()

dummy = torch.randn(1, 950, 52)

torch.onnx.export(
    student_cpu, dummy,
    'esp32_fallguardnet_v2.onnx',
    input_names=['csi_input'],
    output_names=['logits'],
    opset_version=17,
    dynamic_axes={'csi_input': {0: 'batch'}, 'logits': {0: 'batch'}}
)

# Save results JSON
results = {
    'model': 'FallGuardNet_v2_distilled',
    'params': n_params,
    'teacher': 'ResNet18',
    'distillation': {'alpha': 0.7, 'temperature': 4.0},
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1': float(f1),
    'specificity': float(spec),
    'confusion_matrix': cm.tolist(),
    'per_activity_fp_rate': {k: float(v) for k, v in fp_rates.items()}
}

with open('fallguardnet_v2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('✅ Saved:')
print('   • fallguardnet_v2_best.pt        (PyTorch weights)')
print('   • esp32_fallguardnet_v2.onnx      (ONNX for deployment)')
print('   • fallguardnet_v2_results.json    (All metrics)')
print('   • fallguardnet_v2_results.png     (Visualization)')
print(f'\n🏆 FallGuardNet v2: F1={f1:.4f} | Specificity={spec:.4f} | Params={n_params:,}')